In [1]:
import numpy as np
import torch

from src.gym import TradingEnv

In [2]:
def scurve(delta, alpha, beta, delta_0):
    return 1 / (1 + torch.exp(alpha + beta / delta_0 * delta))

def utility(
    ask_delta,
    bid_delta,
    q,
    z,
    ask_lamda,
    bid_lamda,
    gamma,
    alpha,
    beta,
    delta_0,
    kappa,
    delta_t,
    sigma,
):
    t = ask_delta.shape[1]

    res = 0
    for i in range(t):
        res += (
            z
            * ask_delta[:, i]
            * ask_lamda[:, i]
            * scurve(ask_delta[:, i], alpha, beta, delta_0)
            + z
            * bid_delta[:, i]
            * bid_lamda[:, i]
            * scurve(bid_delta[:, i], alpha, beta, delta_0)
            + kappa * (ask_lamda[:, i] - bid_lamda[:, i]) * q[:, i]
            - gamma / 2 * q[:, i] ** 2 * sigma**2
        ) * delta_t
    return torch.mean(res)

In [3]:
from src.model import RLNet

def fair_price(env, gamma, delta_0, z, alpha, beta, batch_size=100, epoch=100):
    ask_model = RLNet(5, 1, [10, 20, 10])
    bid_model = RLNet(5, 1, [10, 20, 10])

    ask_optimzer = torch.optim.Adam(ask_model.parameters(), lr=0.001, betas=(0.9, 0.99))
    bid_optimizer = torch.optim.Adam(
        bid_model.parameters(), lr=0.001, betas=(0.9, 0.99)
    )

    for _ in range(epoch):
        ask_lamda, bid_lamda, prices = env.sample(batch_size)
        ask_lamda = torch.tensor(ask_lamda).to(torch.float32)
        bid_lamda = torch.tensor(bid_lamda).to(torch.float32)
        prices = torch.tensor(prices).to(torch.float32)
        q = torch.zeros((batch_size, env.sp.num_time_interval + 1))
        ask_delta = torch.zeros((batch_size, env.sp.num_time_interval))
        bid_delta = torch.zeros((batch_size, env.sp.num_time_interval))
        for t in range(env.sp.num_time_interval):
            ask_optimzer.zero_grad()
            bid_optimizer.zero_grad()
            q[:, t + 1] = z * (bid_lamda[:, t] - ask_lamda[:, t]) * env.sp.delta_t
            input = torch.cat(
                (
                    torch.ones((batch_size, 1)) * t * env.sp.delta_t,
                    q[:, t : t + 1],
                    ask_lamda[:, t : t + 1],
                    bid_lamda[:, t : t + 1],
                    prices[:, t : t + 1],
                ),
                dim=1,
            )
            ask_delta[:, t : t + 1] = ask_model(input)
            bid_delta[:, t : t + 1] = bid_model(input)

        loss = utility(
            ask_delta,
            bid_delta,
            q,
            z,
            ask_lamda,
            bid_lamda,
            gamma,
            alpha,
            beta,
            delta_0,
            env.sp.k,
            env.sp.delta_t,
            env.sp.sigma,
        )
        loss.backward()
        ask_optimzer.step()
        bid_optimizer.step()

        print(f"Loss: {round(loss.item(), 3)}")

In [4]:
Q = np.array(
    [
        [-14.01, 4.37, 4.37, 5.27],
        [19.32, -60.91, 12.54, 29.05],
        [19.32, 12.54, -60.91, 29.05],
        [23.67, 15.00, 15.00, -53.67],
    ]
)

In [9]:
basic = {"dim": 1, "total_time": 30, "num_time_interval": 30}
specific = {
    "init": 103.593,
    "sigma": 0.1839 / np.sqrt(252),
    "nls": 2,
    "lamdas": np.array([10.83, 73.03]) / 10,
    "init_state": 2,
    "k": 2.29,
    "Q": Q,
}

env = TradingEnv(basic, specific)

In [6]:
env.sample(10)[1].shape

(10, 30)

In [10]:
env.sample(10)

(array([[7.303, 1.083, 1.083, 1.083, 7.303, 7.303, 1.083, 1.083, 7.303,
         1.083, 7.303, 1.083, 1.083, 7.303, 1.083, 1.083, 1.083, 1.083,
         1.083, 7.303, 7.303, 1.083, 7.303, 7.303, 1.083, 1.083, 1.083,
         1.083, 1.083, 1.083],
        [7.303, 1.083, 7.303, 1.083, 1.083, 1.083, 1.083, 7.303, 7.303,
         7.303, 1.083, 1.083, 1.083, 7.303, 1.083, 7.303, 7.303, 1.083,
         7.303, 1.083, 1.083, 7.303, 7.303, 1.083, 1.083, 1.083, 1.083,
         1.083, 1.083, 7.303],
        [7.303, 7.303, 1.083, 1.083, 1.083, 7.303, 1.083, 1.083, 1.083,
         1.083, 1.083, 1.083, 1.083, 7.303, 1.083, 1.083, 1.083, 1.083,
         7.303, 1.083, 7.303, 1.083, 1.083, 1.083, 1.083, 1.083, 1.083,
         1.083, 1.083, 7.303],
        [7.303, 1.083, 7.303, 7.303, 1.083, 7.303, 1.083, 7.303, 1.083,
         7.303, 1.083, 7.303, 1.083, 7.303, 1.083, 1.083, 1.083, 7.303,
         1.083, 1.083, 1.083, 7.303, 7.303, 1.083, 1.083, 1.083, 1.083,
         1.083, 1.083, 1.083],
        [7.3

In [7]:
fair_price(env, 0.0005, 0.09, 1, -0.7, 3.1)

Loss: -63.247
Loss: -65.444
Loss: -39.612
Loss: -43.875
Loss: -58.038
Loss: -54.103
Loss: -48.237
Loss: -48.004
Loss: -69.228
Loss: -55.338
Loss: -55.759
Loss: -47.172
Loss: -48.463
Loss: -85.995
Loss: -89.726
Loss: -61.298
Loss: -62.014
Loss: -79.895
Loss: -71.475
Loss: -99.005
Loss: -73.871
Loss: -71.603
Loss: -80.86
Loss: -93.598
Loss: -114.08
Loss: -81.717
Loss: -80.448
Loss: -87.236
Loss: -90.282
Loss: -64.664
Loss: -106.629
Loss: -117.312
Loss: -111.909
Loss: -124.847
Loss: -125.792
Loss: -102.364
Loss: -106.826
Loss: nan
Loss: nan
Loss: nan
Loss: nan
Loss: nan
Loss: nan
Loss: nan
Loss: nan
Loss: nan
Loss: nan
Loss: nan
Loss: nan
Loss: nan


KeyboardInterrupt: 